<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


## Extract the flight tables from Wikipedia
The bundled HTML contains the flight tables from revision 1027686922. Use the numbered Falcon 9 rows, as in the course exercise. Keep the first payload entry shown on each numbered row. Preserve source text for payload mass when a value is unknown or qualified.

In [1]:
from pathlib import Path
import re
import pandas as pd
from bs4 import BeautifulSoup

soup = BeautifulSoup(Path("data/wikipedia-launch-tables.html").read_text(), "html.parser")
print(soup.title.get_text(), len(soup.select("table.wikitable")), "launch tables")

List of Falcon 9 and Falcon Heavy launches - Wikipedia 9 launch tables


In [2]:
fields = ["Date and time (UTC)", "Booster version", "Launch site", "Payload",
          "Payload mass", "Orbit", "Customer", "Launch outcome", "Booster landing"]
records = []
for table in soup.select("table.wikitable"):
    for row in table.select("tr"):
        number = row.find("th", recursive=False)
        if number is None or not number.get_text(strip=True).isdigit():
            continue
        cells = row.find_all("td", recursive=False)
        assert len(cells) == len(fields)
        for cell in cells:
            for reference in cell.select("sup.reference"):
                reference.decompose()
        record = dict(zip(fields, [cell.get_text(" ", strip=True) or None for cell in cells]))
        record["Flight No."] = int(number.get_text(strip=True))
        records.append(record)
df = pd.DataFrame(records).set_index("Flight No.").sort_index()
assert list(df.index) == list(range(1, 122))
df.head()

Out[0]: 
               Date and time (UTC)  ...      Booster landing
Flight No.                          ...                     
1               4 June 2010, 18:45  ...  Failure (parachute)
2           8 December 2010, 15:43  ...  Failure (parachute)
3               22 May 2012, 07:44  ...           No attempt
4            8 October 2012, 00:35  ...           No attempt
5              1 March 2013, 15:10  ...           No attempt

[5 rows x 9 columns]


,Date and time (UTC),Booster version,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
Flight No.,,,,,,,,,
1,"4 June 2010, 18:45",F9 v1.0 B0003.1,"CCAFS , SLC-40",Dragon Spacecraft Qualification Unit,None,LEO,SpaceX,Success,Failure (parachute)
2,"8 December 2010, 15:43",F9 v1.0 B0004.1,"CCAFS , SLC-40",Dragon demo flight C1 (Dragon C101),None,LEO ( ISS ),NASA ( COTS ) NRO,Success,Failure (parachute)
3,"22 May 2012, 07:44",F9 v1.0 B0005.1,"CCAFS , SLC-40",Dragon demo flight C2+ (Dragon C102),"525 kg (1,157 lb)",LEO ( ISS ),NASA ( COTS ),Success,No attempt
4,"8 October 2012, 00:35",F9 v1.0 B0006.1,"CCAFS , SLC-40",SpaceX CRS-1 (Dragon C103),"4,700 kg (10,400 lb)",LEO ( ISS ),NASA ( CRS ),Success,No attempt
5,"1 March 2013, 15:10",F9 v1.0 B0007.1,"CCAFS , SLC-40",SpaceX CRS-2 (Dragon C104),"4,877 kg (10,752 lb)",LEO ( ISS ),NASA ( CRS ),Success,No attempt


In [3]:
# Retain the original UTC text as well as separate date and time fields.
parts = df["Date and time (UTC)"].str.extract(r"(\d{1,2} [A-Za-z]+ \d{4}).*?(\d{1,2}:\d{2})")
parts.columns = ["Date", "Time"]
df = df.join(parts)
df.to_csv("spacex_web_scraped.csv")
print(f"{len(df)} numbered flight records exported")
df[["Date", "Time", "Launch site", "Payload mass"]].tail()

121 numbered flight records exported
Out[0]: 
                   Date   Time     Launch site             Payload mass
Flight No.                                                             
117          9 May 2021  06:42  CCSFS , SLC-40    15,600 kg (34,400 lb)
118         15 May 2021  22:56    KSC , LC-39A  ~ 14,000 kg (31,000 lb)
119         26 May 2021  18:59  CCSFS , SLC-40    15,600 kg (34,400 lb)
120         3 June 2021  17:29    KSC , LC-39A      3,328 kg (7,337 lb)
121         6 June 2021  04:26  CCSFS , SLC-40     7,000 kg (15,000 lb)


,Date,Time,Launch site,Payload mass
Flight No.,,,,
117,9 May 2021,06:42,"CCSFS , SLC-40","15,600 kg (34,400 lb)"
118,15 May 2021,22:56,"KSC , LC-39A","~ 14,000 kg (31,000 lb)"
119,26 May 2021,18:59,"CCSFS , SLC-40","15,600 kg (34,400 lb)"
120,3 June 2021,17:29,"KSC , LC-39A","3,328 kg (7,337 lb)"
121,6 June 2021,04:26,"CCSFS , SLC-40","7,000 kg (15,000 lb)"


## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
